# Extend Binance 1m Perps Data

Fetches minute-level perpetual OHLCV data from Binance API for **2026-02-14 to 2026-02-22**
and appends it to the existing parquet files in `my_strategies/data/binance_1m_perps_20241201_20260213/`.

In [1]:
import time
from pathlib import Path

import pandas as pd
from binance.client import Client
from binance.enums import HistoricalKlinesType

In [2]:
# --- Configuration ---
FETCH_START = "2026-02-14"  # day after existing data ends
FETCH_END = "2026-02-22"    # last full day to include
DATA_DIR = Path("../data/binance_1m_perps_20241201_20260213")

# Rate-limit pause between symbols (seconds)
SLEEP_BETWEEN_SYMBOLS = 1.0

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Fetch range: {FETCH_START} -> {FETCH_END}")

Data dir: /home/ra_yeye/2026_projects/nautilus_trader/my_strategies/data/binance_1m_perps_20241201_20260213
Fetch range: 2026-02-14 -> 2026-02-22


In [3]:
# Discover symbols from existing parquet files
existing_files = sorted(DATA_DIR.glob("*.parquet"))
symbols = [f.stem for f in existing_files]
print(f"Found {len(symbols)} existing symbol files")
print(symbols[:10], "...")

Found 121 existing symbol files
['0GUSDT', 'AAVEUSDT', 'ADAUSDT', 'AIXBTUSDT', 'ALGOUSDT', 'ALTUSDT', 'ANIMEUSDT', 'APTUSDT', 'ARBUSDT', 'ARUSDT'] ...


In [4]:
def fetch_perp_klines(client: Client, symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    """Fetch 1-minute perpetual futures klines from Binance for a single symbol."""
    klines = client.get_historical_klines(
        symbol=symbol,
        interval=Client.KLINE_INTERVAL_1MINUTE,
        start_str=f"{start_date} 00:00:00",
        end_str=f"{end_date} 23:59:59",
        klines_type=HistoricalKlinesType.FUTURES,
    )

    if not klines:
        return pd.DataFrame()

    df = pd.DataFrame(klines, columns=[
        "timestamp", "open", "high", "low", "close", "volume",
        "close_time", "quote_volume", "trades_count", "taker_buy_volume",
        "taker_buy_quote_volume", "ignore",
    ])

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms", utc=True)

    for col in ["open", "high", "low", "close", "volume", "quote_volume",
                "taker_buy_volume", "taker_buy_quote_volume"]:
        df[col] = df[col].astype(float)
    df["trades_count"] = df["trades_count"].astype(int)

    df["symbol"] = symbol
    df["interval"] = "1m"

    df = df[[
        "symbol", "interval", "timestamp", "close_time",
        "open", "high", "low", "close", "volume", "quote_volume",
        "taker_buy_volume", "taker_buy_quote_volume", "trades_count",
    ]].reset_index(drop=True)

    return df

In [5]:
# Fetch new data and append to existing parquet files
client = Client()

successful = 0
failed = 0
skipped = 0
failed_symbols = []

for i, symbol in enumerate(symbols, 1):
    file_path = DATA_DIR / f"{symbol}.parquet"
    try:
        print(f"[{i}/{len(symbols)}] {symbol}...", end=" ", flush=True)
        t0 = time.time()

        # Fetch new bars
        new_df = fetch_perp_klines(client, symbol, FETCH_START, FETCH_END)

        if len(new_df) == 0:
            print("no new data")
            skipped += 1
            continue

        # Read existing data
        old_df = pd.read_parquet(file_path)

        # Filter out any overlap (new rows must be strictly after existing data)
        last_existing_ts = old_df["timestamp"].max()
        new_df = new_df[new_df["timestamp"] > last_existing_ts]

        if len(new_df) == 0:
            print("no new rows after dedup")
            skipped += 1
            continue

        # Concatenate and save
        combined = pd.concat([old_df, new_df], ignore_index=True)
        combined.to_parquet(file_path, index=False)

        elapsed = time.time() - t0
        print(
            f"+{len(new_df):,} rows -> {len(combined):,} total  "
            f"({elapsed:.1f}s)  "
            f"[{combined['timestamp'].min().date()} -> {combined['timestamp'].max().date()}]"
        )
        successful += 1

    except Exception as e:
        print(f"FAILED: {e}")
        failed += 1
        failed_symbols.append(symbol)

    if i < len(symbols):
        time.sleep(SLEEP_BETWEEN_SYMBOLS)

print("\n" + "=" * 60)
print(f"Successful: {successful}  |  Skipped: {skipped}  |  Failed: {failed}")
if failed_symbols:
    print(f"Failed symbols: {', '.join(failed_symbols)}")
print("=" * 60)

[1/121] 0GUSDT... +12,960 rows -> 228,015 total  (6.2s)  [2025-09-17 -> 2026-02-22]
[2/121] AAVEUSDT... +12,960 rows -> 646,560 total  (8.0s)  [2024-12-01 -> 2026-02-22]
[3/121] ADAUSDT... +12,960 rows -> 646,560 total  (9.0s)  [2024-12-01 -> 2026-02-22]
[4/121] AIXBTUSDT... +12,960 rows -> 618,089 total  (8.0s)  [2024-12-20 -> 2026-02-22]
[5/121] ALGOUSDT... +12,960 rows -> 646,560 total  (8.1s)  [2024-12-01 -> 2026-02-22]
[6/121] ALTUSDT... +12,960 rows -> 646,560 total  (8.4s)  [2024-12-01 -> 2026-02-22]
[7/121] ANIMEUSDT... +12,960 rows -> 569,220 total  (7.8s)  [2025-01-23 -> 2026-02-22]
[8/121] APTUSDT... +12,960 rows -> 646,560 total  (8.9s)  [2024-12-01 -> 2026-02-22]
[9/121] ARBUSDT... +12,960 rows -> 646,560 total  (8.8s)  [2024-12-01 -> 2026-02-22]
[10/121] ARUSDT... +12,960 rows -> 646,560 total  (7.9s)  [2024-12-01 -> 2026-02-22]
[11/121] ASTERUSDT... +12,960 rows -> 225,360 total  (6.5s)  [2025-09-19 -> 2026-02-22]
[12/121] AVAXUSDT... +12,960 rows -> 646,560 total  (8.3s

In [6]:
# Verify: spot-check a symbol
check = pd.read_parquet(DATA_DIR / "BTCUSDT.parquet")
print(f"BTCUSDT shape: {check.shape}")
print(f"Date range: {check['timestamp'].min()} -> {check['timestamp'].max()}")
print(f"Dtypes:\n{check.dtypes}")
print(f"\nLast 3 rows:\n{check.tail(3)}")

BTCUSDT shape: (646560, 13)
Date range: 2024-12-01 00:00:00+00:00 -> 2026-02-22 23:59:00+00:00
Dtypes:
symbol                                 object
interval                               object
timestamp                 datetime64[us, UTC]
close_time                datetime64[us, UTC]
open                                  float64
high                                  float64
low                                   float64
close                                 float64
volume                                float64
quote_volume                          float64
taker_buy_volume                      float64
taker_buy_quote_volume                float64
trades_count                            int64
dtype: object

Last 3 rows:
         symbol interval                 timestamp  \
646557  BTCUSDT       1m 2026-02-22 23:57:00+00:00   
646558  BTCUSDT       1m 2026-02-22 23:58:00+00:00   
646559  BTCUSDT       1m 2026-02-22 23:59:00+00:00   

                             close_time     open     h